In [1]:
!pip install -q datasets huggingface_hub scikit-learn pandas

In [2]:
import pandas as pd

from datasets import load_dataset
from huggingface_hub import login
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")
login(token=HF_TOKEN)

print("Logged in.")

Enter your Hugging Face token: ··········
Logged in.


In [3]:
GPT_OSS_REPO_ID = (
    "dinushiTJ/"
    "nz_research_commons_gpt_oss_120b_openrouter_results_1200"
)

gpt_dataset = load_dataset(
    GPT_OSS_REPO_ID,
    split="train",
)

df = gpt_dataset.to_pandas()

print("Rows loaded:", len(df))
print("Columns:")
print(df.columns.tolist())

README.md:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.60MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Rows loaded: 1200
Columns:
['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'conversations', 'label_text', 'row_index', 'gpt_oss_120b_model', 'gpt_oss_120b_label', 'gpt_oss_120b_confidence', 'gpt_oss_120b_reason', 'gpt_oss_120b_raw_output', 'gpt_oss_120b_error', 'seconds']


In [8]:
GEMINI_LABEL_COL = "label_text"
GPT_OSS_120B_LABEL_COL = "gpt_oss_120b_label"

print(
    "Gemini label column exists:",
    GEMINI_LABEL_COL in df.columns,
)

print(
    "GPT-OSS 120B label column exists:",
    GPT_OSS_120B_LABEL_COL in df.columns,
)

print("\nGemini labels:")
print(df[GEMINI_LABEL_COL].value_counts(dropna=False))

print("\nGPT-OSS 120B labels:")
print(df[GPT_OSS_120B_LABEL_COL].value_counts(dropna=False))

Gemini label column exists: True
GPT-OSS 120B label column exists: True

Gemini labels:
label_text
non_maori_origin    600
maori_origin        600
Name: count, dtype: int64

GPT-OSS 120B labels:
gpt_oss_120b_label
non_maori_origin    716
maori_origin        484
Name: count, dtype: int64


In [9]:
VALID_LABELS = [
    "maori_origin",
    "non_maori_origin",
]

comparison_df = df[
    df[GEMINI_LABEL_COL].isin(VALID_LABELS)
    & df[GPT_OSS_120B_LABEL_COL].isin(VALID_LABELS)
].copy()

comparison_df["labels_match"] = (
    comparison_df[GEMINI_LABEL_COL]
    == comparison_df[GPT_OSS_120B_LABEL_COL]
)

total_compared = len(comparison_df)
matching_count = int(
    comparison_df["labels_match"].sum()
)
different_count = (
    total_compared - matching_count
)

agreement_rate = (
    matching_count / total_compared
    if total_compared > 0
    else 0
)

print("Total records loaded:", len(df))
print("Total records compared:", total_compared)
print("Matching labels:", matching_count)
print("Different labels:", different_count)
print("Agreement rate:", f"{agreement_rate:.2%}")

Total records loaded: 1200
Total records compared: 1200
Matching labels: 1082
Different labels: 118
Agreement rate: 90.17%


In [10]:
confusion_matrix = pd.crosstab(
    comparison_df[GEMINI_LABEL_COL],
    comparison_df[GPT_OSS_120B_LABEL_COL],
    rownames=["Gemini label"],
    colnames=["GPT-OSS 120B label"],
    margins=True,
)

display(confusion_matrix)

GPT-OSS 120B label,maori_origin,non_maori_origin,All
Gemini label,,,
maori_origin,483,117,600
non_maori_origin,1,599,600
All,484,716,1200


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    cohen_kappa_score,
)

y_true = comparison_df[GEMINI_LABEL_COL]
y_pred = comparison_df[GPT_OSS_120B_LABEL_COL]

accuracy = accuracy_score(
    y_true,
    y_pred,
)

precision = precision_score(
    y_true,
    y_pred,
    pos_label="maori_origin",
    zero_division=0,
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label="maori_origin",
    zero_division=0,
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label="maori_origin",
    zero_division=0,
)

kappa = cohen_kappa_score(
    y_true,
    y_pred,
)

print("Accuracy:", f"{accuracy:.2%}")
print(
    "Precision — Māori origin:",
    f"{precision:.2%}",
)
print(
    "Recall — Māori origin:",
    f"{recall:.2%}",
)
print(
    "F1 score — Māori origin:",
    f"{f1:.2%}",
)
print("Cohen's kappa:", round(kappa, 4))

print("\nClassification report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        zero_division=0,
    )
)

Accuracy: 90.17%
Precision — Māori origin: 99.79%
Recall — Māori origin: 80.50%
F1 score — Māori origin: 89.11%
Cohen's kappa: 0.8033

Classification report:

                  precision    recall  f1-score   support

    maori_origin       1.00      0.81      0.89       600
non_maori_origin       0.84      1.00      0.91       600

        accuracy                           0.90      1200
       macro avg       0.92      0.90      0.90      1200
    weighted avg       0.92      0.90      0.90      1200



In [13]:
disagreement_df = comparison_df[
    ~comparison_df["labels_match"]
].copy()

print(
    "Number of disagreements:",
    len(disagreement_df),
)

columns_to_show = [
    "row_index",
    "record_id_hash",
    "title",
    GEMINI_LABEL_COL,
    GPT_OSS_120B_LABEL_COL,
    "gpt_oss_120B_confidence",
    "gpt_oss_120B_reason",
]

columns_to_show = [
    column
    for column in columns_to_show
    if column in disagreement_df.columns
]

display(
    disagreement_df[
        columns_to_show
    ]
)

Number of disagreements: 118


,row_index,record_id_hash,title,label_text,gpt_oss_120b_label
4,4,4f637ba8200ed359c2b15f73989dc3323a6ca026106b7a...,"An old story, a new voice: Tertiary education ...",maori_origin,non_maori_origin
8,8,1827710b6ec67816e6b0e0fcb4f155411edba44944608a...,2016 Subtidal Ecological Survey of Tauranga Ha...,maori_origin,non_maori_origin
37,37,b79f6bb27003a9d4caabcf59d63f2beb92232c7f8791ef...,Prevalence of sociodemographic factors in a co...,maori_origin,non_maori_origin
41,41,1748aee60bceb198399d93c93a4c0ff6ca2e555f9704fb...,Indigenous Rights Hollow Rights?,maori_origin,non_maori_origin
45,45,2aaa287a6a432683d2dbe0082ee269ca1fdec901643625...,Low copper levels measured in the aortic wall ...,maori_origin,non_maori_origin
...,...,...,...,...,...
1158,1158,dbb533df6b58d0942715fddbd09c70be3ff708a463cce1...,Land/seascapes of exclusion: social order on t...,maori_origin,non_maori_origin
1161,1161,30d3eee7f0cc8df1daa315035ca924ab7318dc33d87f4d...,Personality and Representation in Environmenta...,maori_origin,non_maori_origin
1163,1163,ab1131fbfe7d74a955e746e62d877bcc69ecbf08aa25d8...,Co-constructions of gender and ethnicity in Ne...,maori_origin,non_maori_origin
1182,1182,9cba65b2e49a44c620add659af8a3aefe172c1b5130bab...,An Interactive User Management System for Mult...,maori_origin,non_maori_origin
